In [1]:
# Plotting gridpoint PM2.5 mortality for one model, ensemble mean, final 10 years
# Log scale which starts at zero

In [2]:
import os
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import SymLogNorm
from matplotlib.ticker import FuncFormatter
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from utils.utils import get_scenario_config
from utils.utils import autosize_figure, land_filter

In [7]:
def load_mortality_ensemble_members(model, scenario, config):
    # === Path config ===
    MORT_DIR = f"/glade/work/awells/air_quality/{model}/mortality/pm25/gridpoint_mortality/"

    ensemble_members = config["ensemble_members"]
    years = config["years"]
    dates = f"{years.start}-{years.stop}"

    ensemble = []
    for ens_num in ensemble_members:
        print(f"Processing ensemble number {ens_num:02d}")
        file = f"Mortality_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
        path = os.path.join(MORT_DIR, file)
        da = xr.open_dataarray(path)
        ensemble.append(da)

    new_da = xr.concat(
        ensemble,
        dim=xr.DataArray(
            np.arange(1, len(ensemble)+1),
            dims="ensemble",
            name="ensemble"))

    return new_da

In [4]:
def process_mortality_for_plotting(model, scenario, config):
    """
    Calculate final decade and ensemble mean
    """

    # Load ensemble members
    da = load_mortality_ensemble_members(model, scenario, config)

    # Calculate end of scenario mean (10 years)
    end_idx = da.sizes["year"]
    # Select the last 10 years
    da_last10 = da.isel(year=slice(end_idx - 10, end_idx))

    # Get the first and last time values
    start_time = da_last10.year[0].item()
    end_time = da_last10.year[-1].item()

    # Calculate ensemble and temporal mean
    da_mean = da_last10.mean(dim=("ensemble", "year"))
    return da_mean, start_time, end_time

In [5]:
def plot_gridpoint_mortality(da, model, scenario, start_year, end_year, SAVE_DIR):
    # Create figure
    fig = plt.figure(figsize=autosize_figure(1, 1))
    gs = GridSpec(2, 1, height_ratios=[15, 1])  # rows, columns

    projection = ccrs.Robinson()
    crs = ccrs.PlateCarree()

    country_borders = cfeature.NaturalEarthFeature(
        category='cultural',
        name='admin_0_boundary_lines_land',
        scale='50m',
        facecolor='none')

    # === Use log scale for normalization ===
    vmax = da.max().item()
    cmap = plt.get_cmap("viridis")

    # Define a linear threshold region around zero
    linthresh = 0.0001

    norm = SymLogNorm(linthresh=linthresh, vmin=0, vmax=vmax, base=10)

    # First panel
    ax = fig.add_subplot(gs[0, 0], projection=projection, frameon=True)
    cb = land_filter(da).plot(
        transform=crs,
        add_colorbar=False,
        cmap=cmap,
        norm=norm,
        subplot_kws={'projection': projection}
    )
    ax.coastlines(resolution="50m", linewidth=0.75)
    ax.add_feature(country_borders, edgecolor='k', linewidth=0.75)
    plt.title(f"Mortality due to PM2.5\n {model} {scenario} {start_year}-{end_year}", fontsize=16)

    # First colorbar
    cax = fig.add_subplot(gs[1, 0])
    col_bar = plt.colorbar(cb, cax=cax, orientation='horizontal')
    col_bar.set_label("Mortality", fontsize=13)

    # Ticks at 1, 2, 5 × powers of 10
    formatter = FuncFormatter(lambda v, _: f"{v:g}")
    col_bar.ax.xaxis.set_major_formatter(formatter)

    plt.tight_layout()

    out_file = f"Mortality_gridpoint_{model}_{scenario}_{start_year}-{end_year}.png"
    out_path = os.path.join(SAVE_DIR, out_file)
    plt.savefig(out_path)
    return

In [10]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
SAVE_DIR = "/glade/u/home/awells/air_quality_project/plotting/mortality/pm25/Gridpoint/"

model = "CESM2"
scenario = "G6-1.5K"

config = get_scenario_config(model, scenario)

da, first_year, last_year = process_mortality_for_plotting(model, scenario, config)

# Plotting
plot_gridpoint_mortality(da, model, scenario, first_year, last_year, SAVE_DIR)

Processing ensemble number 01


FileNotFoundError: [Errno 2] No such file or directory: '/glade/work/awells/air_quality/UKESM1/mortality/pm25/gridpoint_mortality/Mortality_UKESM1_G6-1.5K_01_2035-2084.nc'